# 03 — Validación Cross-Gene
**EnergyFingerprint Pipeline** | Notebook 3 de 3

¿Los patrones aprendidos por la CNN en un gen se transfieren a otro?

**Experimentos**:
- **Zero-shot**: entrenar en gen A, predecir gen B sin reentrenamiento
- **Intra-gene**: CV en gen B como referencia
- **Comparación 9ch vs 11ch**: ¿el canal ESM-1v mejora la transferencia?

**Entrada**: tensores `.npz` de ambos genes (generados por notebook 01)

## Configuración

In [ ]:
# ══════════════════════════════════════════════════
# CONFIGURAR AQUÍ
# ══════════════════════════════════════════════════
GENE_TRAIN  = "brca1"       # gen de entrenamiento
GENE_TEST   = "scn1a"        # gen de test (zero-shot)
SUBSET      = "likely"      # "likely" o "all"
COMPARE_9_11 = False         # comparar 9ch vs 11ch
EPOCHS      = 80            # épocas entrenamiento zero-shot
CV_EPOCHS   = 60            # épocas intra-gene CV
N_FOLDS     = 3             # folds intra-gene
N_REPS      = 3             # repeticiones zero-shot
SEED        = 42
# ══════════════════════════════════════════════════

## Imports

In [ ]:
import sys, os, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, roc_curve
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

from energyfingerprint.model import (
    EnergySignalCNN, VariantDataset,
    EarlyStopping, create_weighted_sampler
)

def get_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print(f"Train: {GENE_TRAIN.upper()} → Test: {GENE_TEST.upper()} | Device: {device}")

## Funciones auxiliares

In [ ]:
def normalize_tensors(t, ref_stats=None):
    t_norm = t.copy()
    stats = {}
    for c in range(t.shape[2]):
        if ref_stats and c in ref_stats:
            mu, sigma = ref_stats[c]["mean"], ref_stats[c]["std"]
        else:
            d = t[:, :, c].flatten()
            mu, sigma = d.mean(), d.std()
        if sigma > 1e-10:
            t_norm[:, :, c] = (t_norm[:, :, c] - mu) / sigma
        stats[c] = {"mean": float(mu), "std": float(sigma)}
    return t_norm, stats

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total, n = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total += loss.item() * len(y)
        n += len(y)
    return total / n

def eval_model(model, loader, device):
    model.eval()
    probs_all, labels_all = [], []
    with torch.no_grad():
        for x, y in loader:
            probs = torch.softmax(model(x.to(device)), dim=1)[:, 1].cpu().numpy()
            probs_all.extend(probs)
            labels_all.extend(y.numpy())
    l, p = np.array(labels_all), np.array(probs_all)
    preds = (p >= 0.5).astype(int)
    auc = roc_auc_score(l, p) if len(np.unique(l)) > 1 else 0.5
    return {"auc": auc, "f1": f1_score(l, preds, zero_division=0),
            "acc": accuracy_score(l, preds), "probs": p, "labels": l}

def train_full(tensors, labels, n_ch, device, epochs=80, seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
    ds = VariantDataset(tensors, labels, augment=True)
    sampler = create_weighted_sampler(labels)
    loader = DataLoader(ds, batch_size=32, sampler=sampler, num_workers=0)
    model = EnergySignalCNN(n_channels=n_ch, n_classes=2, dropout=0.2, n_heads=4).to(device)
    cc = np.bincount(labels)
    cw = torch.FloatTensor([len(labels) / (2*c) for c in cc]).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    for ep in range(epochs):
        train_epoch(model, loader, criterion, optimizer, device)
    return model

def train_cv(tensors, labels, n_ch, device, n_folds=3, epochs=60, seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    aucs, f1s = [], []
    all_p, all_l = [], []
    for fi, (tri, vi) in enumerate(skf.split(tensors, labels)):
        tr_ds = VariantDataset(tensors[tri], labels[tri], augment=True)
        va_ds = VariantDataset(tensors[vi], labels[vi], augment=False)
        sampler = create_weighted_sampler(labels[tri])
        tr_ld = DataLoader(tr_ds, batch_size=32, sampler=sampler, num_workers=0)
        va_ld = DataLoader(va_ds, batch_size=32, shuffle=False, num_workers=0)
        model = EnergySignalCNN(n_channels=n_ch, n_classes=2, dropout=0.2, n_heads=4).to(device)
        cc = np.bincount(labels[tri])
        cw = torch.FloatTensor([len(labels[tri]) / (2*c) for c in cc]).to(device)
        criterion = nn.CrossEntropyLoss(weight=cw)
        optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        es = EarlyStopping(patience=12)
        best_auc, best_st = 0, None
        for ep in range(epochs):
            train_epoch(model, tr_ld, criterion, optimizer, device)
            m = eval_model(model, va_ld, device)
            if m["auc"] > best_auc:
                best_auc = m["auc"]
                best_st = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            if es(1 - m["auc"]): break
        model.load_state_dict(best_st)
        m = eval_model(model, va_ld, device)
        aucs.append(m["auc"]); f1s.append(m["f1"])
        all_p.extend(m["probs"]); all_l.extend(m["labels"])
        print(f"    Fold {fi+1}: AUC={m['auc']:.4f}  F1={m['f1']:.4f}")
    return {"auc_mean": float(np.mean(aucs)), "auc_std": float(np.std(aucs)),
            "f1_mean": float(np.mean(f1s)),
            "all_probs": np.array(all_p), "all_labels": np.array(all_l)}

## 1. Cargar datos de ambos genes

In [ ]:
def load_gene_tensors(gene, n_ch, subset):
    data_dir = os.path.join(PROJECT_ROOT, "studies", gene, "data")
    suffix = f"_tensors_{n_ch}ch"
    if subset != "all": suffix += f"_{subset}"
    path = os.path.join(data_dir, f"{gene}{suffix}.npz")

    if not os.path.exists(path):
        # Fallback 1: nombre legado (brca1_tensors_11ch_likely.npz)
        legacy = os.path.join(data_dir, f"{gene}_tensors_{n_ch}ch_{subset}.npz")
        if os.path.exists(legacy):
            path = legacy
        else:
            # Fallback 2: brca1_tensors.npz (original 9ch sin sufijo)
            plain = os.path.join(data_dir, f"{gene}_tensors.npz")
            if os.path.exists(plain) and n_ch == 9:
                path = plain
                print(f"  Usando archivo legado: {os.path.basename(path)}")
            else:
                # Fallback 3: glob
                import glob
                candidates = glob.glob(os.path.join(data_dir, f"{gene}_tensors*{n_ch}ch*.npz"))
                if not candidates:
                    candidates = glob.glob(os.path.join(data_dir, f"{gene}_tensors*.npz"))
                if candidates:
                    path = sorted(candidates)[-1]
                    print(f"  Fallback: usando {os.path.basename(path)}")
                else:
                    raise FileNotFoundError(
                        f"No se encuentran tensores para {gene} en {data_dir}. "
                        f"Ejecuta notebook 01 con GENE='{gene}', N_CHANNELS={n_ch}")

    data = np.load(path, allow_pickle=True)
    t, l = data["tensors"], data["labels"]
    print(f"  {gene.upper()}: {t.shape} | {(l==1).sum()}P / {(l==0).sum()}B | {os.path.basename(path)}")
    return t, l

# Ejecutar el experimento para cada configuración de canales
channel_configs = [11, 9] if COMPARE_9_11 else [11]

results_all = {}

for n_ch in channel_configs:
    print(f"\n{'='*60}")
    print(f"Cargando datos con {n_ch} canales")
    print(f"{'='*60}")

    try:
        t_train, l_train = load_gene_tensors(GENE_TRAIN, n_ch, SUBSET)
        t_test,  l_test  = load_gene_tensors(GENE_TEST,  n_ch, SUBSET)
    except FileNotFoundError as e:
        print(f"  ERROR: {e}")
        continue

    # Ajustar canales si difieren
    actual_ch = min(t_train.shape[2], t_test.shape[2])
    if actual_ch < n_ch:
        print(f"  Ajustando a {actual_ch} canales (mínimo disponible)")
        t_train = t_train[:, :, :actual_ch]
        t_test  = t_test[:, :, :actual_ch]
        n_ch = actual_ch

    # Normalizar (train stats como referencia para zero-shot)
    t_train_n, train_stats = normalize_tensors(t_train)
    t_test_zs, _ = normalize_tensors(t_test, ref_stats=train_stats)
    t_test_n,  _ = normalize_tensors(t_test)

    results_all[n_ch] = {
        "t_train_n": t_train_n, "l_train": l_train,
        "t_test_zs": t_test_zs, "t_test_n": t_test_n, "l_test": l_test,
        "n_ch": n_ch,
    }
    print(f"  Datos listos para {n_ch}ch")

## 2. Zero-Shot Transfer

In [ ]:
for n_ch, rd in results_all.items():
    print(f"\n{'='*60}")
    print(f"ZERO-SHOT {GENE_TRAIN.upper()}→{GENE_TEST.upper()} ({n_ch}ch)")
    print(f"{'='*60}")

    test_ds = VariantDataset(rd["t_test_zs"], rd["l_test"], augment=False)
    test_ld = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

    zs_results = []
    for rep in range(N_REPS):
        model = train_full(rd["t_train_n"], rd["l_train"], n_ch, device,
                           epochs=EPOCHS, seed=SEED + rep)
        m = eval_model(model, test_ld, device)
        zs_results.append(m)
        print(f"  Rep {rep+1}: AUC={m['auc']:.4f}  F1={m['f1']:.4f}")

    rd["zs_results"] = zs_results
    rd["zs_auc"] = np.mean([r["auc"] for r in zs_results])
    rd["zs_std"] = np.std([r["auc"] for r in zs_results])
    print(f"\n  >>> Zero-shot AUC: {rd['zs_auc']:.4f} ± {rd['zs_std']:.4f}")

## 3. Intra-Gene (referencia)

In [ ]:
for n_ch, rd in results_all.items():
    print(f"\n{'='*60}")
    print(f"INTRA-GENE {GENE_TEST.upper()} ({n_ch}ch, {N_FOLDS}-fold CV)")
    print(f"{'='*60}")

    ig = train_cv(rd["t_test_n"], rd["l_test"], n_ch, device,
                  n_folds=N_FOLDS, epochs=CV_EPOCHS, seed=SEED)

    rd["ig_results"] = ig
    print(f"\n  >>> Intra-gene AUC: {ig['auc_mean']:.4f} ± {ig['auc_std']:.4f}")

## 4. Resumen comparativo

In [ ]:
print(f"\n{'='*65}")
print(f"RESUMEN CROSS-GENE: {GENE_TRAIN.upper()} → {GENE_TEST.upper()}")
print(f"{'='*65}")
print(f"  Subset: {SUBSET}")
print()

header = f"  {'Config':<20} {'Zero-shot AUC':>15} {'Intra-gene AUC':>16} {'Δ':>8}"
print(header)
print(f"  {'─'*20} {'─'*15} {'─'*16} {'─'*8}")
print(f"  {'Random':<20} {'0.5000':>15} {'0.5000':>16} {'—':>8}")

for n_ch in sorted(results_all.keys()):
    rd = results_all[n_ch]
    zs = rd["zs_auc"]
    ig = rd["ig_results"]["auc_mean"]
    label = f"{n_ch}ch" + (" + ESM" if n_ch >= 11 else " energía")
    print(f"  {label:<20} {zs:>15.4f} {ig:>16.4f} {ig-zs:>+8.4f}")

if len(results_all) == 2:
    ch_list = sorted(results_all.keys())
    zs_delta = results_all[ch_list[1]]["zs_auc"] - results_all[ch_list[0]]["zs_auc"]
    ig_delta = results_all[ch_list[1]]["ig_results"]["auc_mean"] - \
               results_all[ch_list[0]]["ig_results"]["auc_mean"]
    print(f"\n  Efecto ESM-1v:")
    print(f"    Zero-shot: {zs_delta:+.4f}")
    print(f"    Intra-gene: {ig_delta:+.4f}")

## 5. Figuras

In [ ]:
n_configs = len(results_all)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"Cross-Gene: {GENE_TRAIN.upper()} → {GENE_TEST.upper()} ({SUBSET})",
             fontsize=14, fontweight="bold")

colors_map = {9: "#FF9800", 11: "#2196F3"}

# 1. ROC curves
ax = axes[0]
for n_ch, rd in sorted(results_all.items()):
    # Zero-shot (best rep)
    best_r = max(range(N_REPS), key=lambda i: rd["zs_results"][i]["auc"])
    fpr, tpr, _ = roc_curve(rd["zs_results"][best_r]["labels"],
                            rd["zs_results"][best_r]["probs"])
    ax.plot(fpr, tpr, color=colors_map.get(n_ch, "gray"), lw=2,
            label=f"ZS {n_ch}ch: {rd['zs_auc']:.3f}")

    # Intra-gene
    fpr2, tpr2, _ = roc_curve(rd["ig_results"]["all_labels"],
                              rd["ig_results"]["all_probs"])
    ax.plot(fpr2, tpr2, color=colors_map.get(n_ch, "gray"), lw=2, ls="--",
            label=f"IG {n_ch}ch: {rd['ig_results']['auc_mean']:.3f}")

ax.plot([0, 1], [0, 1], "--", color="gray", alpha=0.4)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("ROC Curves")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# 2. Bar comparison
ax = axes[1]
configs = sorted(results_all.keys())
x = np.arange(len(configs))
w = 0.35
zs_vals = [results_all[c]["zs_auc"] for c in configs]
ig_vals = [results_all[c]["ig_results"]["auc_mean"] for c in configs]

bars1 = ax.bar(x - w/2, zs_vals, w, label="Zero-shot", color="#E91E63", alpha=0.8)
bars2 = ax.bar(x + w/2, ig_vals, w, label="Intra-gene", color="#2196F3", alpha=0.8)

for b, v in zip(list(bars1) + list(bars2), zs_vals + ig_vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.3f}",
            ha="center", fontsize=9, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels([f"{c}ch" for c in configs])
ax.set_ylabel("AUC")
ax.set_title("Comparación")
ax.set_ylim(0.3, 1.05)
ax.axhline(y=0.5, color="gray", ls="--", alpha=0.3)
ax.legend()
ax.grid(True, alpha=0.2, axis="y")

# 3. Score distribution (best config)
best_ch = max(results_all.keys())
rd = results_all[best_ch]
best_r = max(range(N_REPS), key=lambda i: rd["zs_results"][i]["auc"])
zb = rd["zs_results"][best_r]

ax = axes[2]
mp = zb["labels"] == 1
mb = zb["labels"] == 0
ax.hist(zb["probs"][mb], bins=20, alpha=0.6, color="#4CAF50",
        label=f"Benign (n={mb.sum()})", density=True)
ax.hist(zb["probs"][mp], bins=20, alpha=0.6, color="#F44336",
        label=f"Pathogenic (n={mp.sum()})", density=True)
ax.axvline(x=0.5, color="gray", ls="--", alpha=0.5)
ax.set_xlabel("P(pathogenic)")
ax.set_ylabel("Density")
ax.set_title(f"Zero-shot scores ({best_ch}ch)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

plt.tight_layout(rect=[0, 0, 1, 0.92])
fig_dir = os.path.join(PROJECT_ROOT, "studies", GENE_TEST, "figures")
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, f"crossgene_{GENE_TRAIN}_{GENE_TEST}.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura guardada: {fig_path}")

## 6. Guardar resultados

In [ ]:
summary = {
    "gene_train": GENE_TRAIN,
    "gene_test": GENE_TEST,
    "subset": SUBSET,
    "configs": {}
}

for n_ch, rd in results_all.items():
    summary["configs"][str(n_ch)] = {
        "zero_shot": {
            "auc_mean": float(rd["zs_auc"]),
            "auc_std": float(rd["zs_std"]),
            "reps": [{"auc": float(r["auc"]), "f1": float(r["f1"])}
                     for r in rd["zs_results"]],
        },
        "intra_gene": {
            "auc_mean": rd["ig_results"]["auc_mean"],
            "auc_std": rd["ig_results"]["auc_std"],
            "f1_mean": rd["ig_results"]["f1_mean"],
        }
    }

data_dir = os.path.join(PROJECT_ROOT, "studies", GENE_TEST, "data")
results_path = os.path.join(data_dir, f"crossgene_{GENE_TRAIN}_{GENE_TEST}.json")
with open(results_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"Resultados guardados: {results_path}")

## Guía de reproducción con otro gen

Para validar con un nuevo gen (ej: `PTEN`):

1. **Crear scripts de descarga** en `studies/pten/scripts/`:
   - `01_download_pten.py` — adaptar de tp53/brca1
   - `03_run_esm.py` — copiar y ajustar paths

2. **Ejecutar descarga y ESM**:
   ```bash
   python studies/pten/scripts/01_download_pten.py
   python studies/pten/scripts/03_run_esm.py
   ```

3. **Notebook 01**: Cambiar `GENE = "pten"`, ejecutar → genera tensores

4. **Notebook 02**: Cambiar `GENE = "pten"`, ejecutar → entrenamiento + ablación

5. **Notebook 03**: Cambiar `GENE_TRAIN / GENE_TEST`, ejecutar → cross-gene

Los únicos archivos requeridos por gen son:
- `studies/<gen>/data/<gen>_cds.fasta`
- `studies/<gen>/data/<gen>_clinvar_missense.csv`
- `studies/<gen>/data/<gen>_esm_scores.csv` (opcional, para 11ch)